###AUTOLOADER

In [0]:
from pyspark.sql import functions as F

import os
import sys

project_pth = os.path.join(os.getcwd(), '..', '..')

sys.path.append(project_pth)



In [0]:
from utils.transformations import reusable

##DimUser

In [0]:
df_user = spark.readStream.format("cloudFiles")\
    .option("cloudFiles.format", "parquet")\
    .option("cloudFiles.schemaLocation", "abfss://silver@junaidazureproject.dfs.core.windows.net/DimUser/checkpoint")\
    .load("abfss://bronze@junaidazureproject.dfs.core.windows.net/DimUser")

In [0]:
df_user = df_user.withColumn("user_name", F.upper(F.col("user_name")))

In [0]:
#/Workspace/Users/junavtar687@gmail.com/spotify_dab/utils/transformations.py
import os
import  sys
print(os.path.join(os.getcwd(),'..','..'))

In [0]:
df_user_obj = reusable()
df_user = df_user_obj.dropColumns(df_user, ['_rescued_data'])
df_user = df_user.dropDuplicates(['user_id'])



In [0]:
(  df_user.writeStream.format("delta")\
                .outputMode("append")\
                .option("checkpointLocation","abfss://silver@junaidazureproject.dfs.core.windows.net/DimUser/checkpoint")\
                .trigger(once = True)\
                .option("path","abfss://silver@junaidazureproject.dfs.core.windows.net/DimUser/data")
                .toTable("spotify_cata.silver.DimUser")

)
         
         

##DimArtist

In [0]:
df_art = spark.readStream.format("cloudFiles")\
    .option("cloudFiles.format", "parquet")\
    .option("cloudFiles.schemaLocation", "abfss://silver@junaidazureproject.dfs.core.windows.net/DimArt/checkpoint")\
    .load("abfss://bronze@junaidazureproject.dfs.core.windows.net/DimArtist")

In [0]:
df_art = reusable().dropColumns(df_art, ['_rescued_data'])

df_art = df_art.dropDuplicates(['artist_id'])

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS spotify_cata.silver;

In [0]:
%skip
(
    df_art.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", "abfss://silver@junaidazureproject.dfs.core.windows.net/DimArt/checkpoint")
    .trigger(once=True)
    .option("path", "abfss://silver@junaidazureproject.dfs.core.windows.net/DimArt/data")
    .toTable("spotify_cata.silver.DimArtist")
)
            

##DimTrack


In [0]:
df_track = spark.readStream.format("cloudFiles")\
    .option("cloudFiles.format", "parquet")\
    .option("cloudFiles.schemaLocation", "abfss://silver@junaidazureproject.dfs.core.windows.net/DimTrack/checkpoint")\
    .load("abfss://bronze@junaidazureproject.dfs.core.windows.net/DimTrack")

In [0]:
df_track = df_track.withColumn("durationFlag", F.when(F.col("duration_sec")< 150 , "low")\
                             .when(F.col("duration_sec")<300 , "medium")\
                             .otherwise("high")) 

df_track = df_track.withColumn("track_name", F.regexp_replace(F.col("track_name"),'-',' '))

df_track = reusable().dropColumns(df_track, ['_rescued_data'])
                        


In [0]:
df_track = reusable().dropColumns(df_track, ['_rescued_data'])

In [0]:
(
    df_track.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", "abfss://silver@junaidazureproject.dfs.core.windows.net/DimTrack/checkpoint")
    .trigger(once=True)
    .option("path", "abfss://silver@junaidazureproject.dfs.core.windows.net/DimTrack/data")
    .toTable("spotify_cata.silver.DimTrack")
)

##DimDate

In [0]:
df_date = spark.readStream.format("cloudFiles")\
    .option("cloudFiles.format", "parquet")\
    .option("cloudFiles.schemaLocation", "abfss://silver@junaidazureproject.dfs.core.windows.net/DimDate/checkpoint")\
    .load("abfss://bronze@junaidazureproject.dfs.core.windows.net/DimDate")

In [0]:
df_date= reusable().dropColumns(df_date, ['_rescued_data'])

(
    df_date.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", "abfss://silver@junaidazureproject.dfs.core.windows.net/DimDate/checkpoint")
    .trigger(once=True)
    .option("path", "abfss://silver@junaidazureproject.dfs.core.windows.net/DimDate/data")
    .toTable("spotify_cata.silver.DimDate")
)


##FactStream

In [0]:
df_fact = spark.readStream.format("cloudFiles")\
    .option("cloudFiles.format", "parquet")\
    .option("cloudFiles.schemaLocation", "abfss://silver@junaidazureproject.dfs.core.windows.net/FactStream/checkpoint")\
    .load("abfss://bronze@junaidazureproject.dfs.core.windows.net/FactStream")

In [0]:
%skip
df_fact = reusable().dropColumns(df_fact, ['_rescued_data'])

(
    df_fact.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", "abfss://silver@junaidazureproject.dfs.core.windows.net/FactStream/checkpoint")
    .trigger(once=True)
    .option("path", "abfss://silver@junaidazureproject.dfs.core.windows.net/FactStream/data")
    .toTable("spotify_cata.silver.FactStream")
)